# 🎯 Module 12.5 - CAPSTONE: Complete End-to-End RL Vision Pipeline

## *Reinforcement Learning for Image Processing - The Grand Finale*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_12_Real_World_Projects/05_Complete_Vision_Pipeline/complete_vision_pipeline.ipynb)

---

This capstone project brings together **everything** from the course into a single, unified system:
an end-to-end vision pipeline where each processing stage is powered by a trained RL agent.

**Pipeline:** Input → Enhancement → Segmentation → Detection → Output

---

## 🎯 Learning Objectives

By completing this capstone, you will:

1. **Architect** a multi-stage RL pipeline for end-to-end vision processing
2. **Formulate** cascaded MDPs with inter-stage state transfer
3. **Implement** RL agents for enhancement, segmentation, and detection
4. **Train** each agent with stage-specific reward functions
5. **Integrate** all stages into a unified, differentiable pipeline
6. **Evaluate** end-to-end performance with ablation studies
7. **Demonstrate** the power of RL-driven computer vision

In [ ]:
!pip install -q numpy matplotlib torch torchvision scipy scikit-learn scikit-image

## Deep Derivation: Complete RL Pipeline for Image Enhancement

### Step 1: State Space Design for Enhancement
$$s_t = \{I_t, h_t, \text{hist}(I_t)\}$$

where $I_t$ is current image, $h_t$ is action history, $\text{hist}(I_t) \in \mathbb{R}^{256}$ is histogram.

**Embedding dimension:** $d = C_{img} \cdot H \cdot W + |\mathcal{A}| \cdot T + 256$

### Step 2: Multi-Objective Reward (Pareto Optimal)
$$R(s_t, a_t) = w_1 \cdot \Delta\text{PSNR} + w_2 \cdot \Delta\text{SSIM} + w_3 \cdot \Delta\text{LPIPS} - w_4 \cdot \text{step\_penalty}$$

**Pareto optimality condition:** No other policy achieves better in ALL objectives simultaneously.

**Scalarization:** With weights $\mathbf{w}$, we solve:
$$\pi^* = \arg\max_\pi \sum_i w_i \cdot J_i(\pi)$$

Different $\mathbf{w}$ trace the Pareto frontier.

### Step 3: Actor-Critic Architecture
**Actor (policy network):**
$$\pi_\theta(a|s) = \mathcal{N}(\mu_\theta(s), \sigma_\theta(s)^2)$$

$$\mu_\theta = \text{MLP}(\text{CNN}(I_t)), \quad \log\sigma_\theta = \text{MLP}(\text{CNN}(I_t))$$

**Critic (value network):**
$$V_\phi(s) = \text{MLP}(\text{CNN}(I_t))$$

### Step 4: PPO Training Loop (Full Algorithm)
For each iteration:
1. Collect $T$ steps: $\{(s_t, a_t, r_t, s_{t+1})\}_{t=0}^T$
2. Compute advantages: $\hat{A}_t = \sum_{l=0}^{T-t} (\gamma\lambda)^l \delta_{t+l}$

where the TD residuals: $\delta_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$

**Derivation of GAE (Generalized Advantage Estimation):**
$$\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$$

This interpolates between:
- $\lambda = 0$: $\hat{A}_t = \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ (TD(0), low variance, high bias)
- $\lambda = 1$: $\hat{A}_t = \sum_{l=0}^\infty \gamma^l r_{t+l} - V(s_t)$ (Monte Carlo, high variance, low bias)

**Proof of bias-variance trade-off:** The bias of GAE:
$$\text{Bias}(\hat{A}^{\text{GAE}}) = \frac{(1-\lambda)\gamma}{1-\gamma\lambda} E[\epsilon_V]$$

where $\epsilon_V = V_\phi(s) - V^*(s)$ is the value function error. As $\lambda \to 1$, bias → 0. The variance:
$$\text{Var}(\hat{A}^{\text{GAE}}) \propto \frac{1}{1 - (\gamma\lambda)^2} \cdot \sigma_r^2$$

As $\lambda \to 0$, variance is minimized. $\blacksquare$

3. For $K$ epochs over minibatches:
   - Compute ratio: $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$
   - Clipped objective: $L^{CLIP} = E\left[\min(r_t \hat{A}_t, \text{clip}(r_t, 1\pm\epsilon)\hat{A}_t)\right]$
   - Update: $\theta \leftarrow \theta + \eta \nabla_\theta L^{CLIP}$

**Derivation of why clipping works:** The clipped objective creates a pessimistic bound:
$$L^{CLIP}(\theta) \leq L^{CPI}(\theta) = E[r_t(\theta)\hat{A}_t]$$

This ensures we never take too large a step, preventing policy collapse. The trust region is implicitly enforced:
$$|r_t - 1| \leq \epsilon \implies D_{KL}(\pi_\theta \| \pi_{\theta_{old}}) \lesssim \epsilon^2 / 2 \quad \blacksquare$$

4. Update critic: $\phi \leftarrow \phi - \eta_v \nabla_\phi \frac{1}{2}\|V_\phi(s_t) - V_t^{\text{target}}\|^2$

### Step 5: Hyperparameter Analysis
**Learning rate schedule (cosine annealing):**
$$\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})\left(1 + \cos\frac{t\pi}{T}\right)$$

**Derivation:** The cosine schedule provides:
- $t = 0$: $\eta_0 = \eta_{max}$ (start with large steps)
- $t = T/2$: $\eta = (\eta_{max} + \eta_{min})/2$ (midpoint)
- $t = T$: $\eta_T = \eta_{min}$ (fine-tuning with small steps)

The smooth decay avoids the abrupt changes of step-decay schedules, leading to more stable training.

**Entropy coefficient decay:**
$$c_2(t) = c_{2,0} \cdot \exp(-t/\tau_H)$$

High entropy early (exploration) → low entropy later (exploitation).

**Derivation of the entropy bonus gradient:**
$$\nabla_\theta H(\pi_\theta) = -\nabla_\theta E_\pi[\log\pi_\theta] = -E_\pi[\nabla_\theta\log\pi_\theta(1 + \log\pi_\theta)]$$

This pushes the policy toward higher entropy (more uniform), counteracting premature convergence. The decay rate $\tau_H$ should be set so that $c_2(T/2) \approx c_{2,0}/10$ — sufficient exploration in the first half of training.

### Step 6: Evaluation Protocol
**Statistical significance testing with paired t-test:**
$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad \text{df} = n-1$$

where $d_i = \text{metric}(\text{RL}) - \text{metric}(\text{baseline})$ for test image $i$.

**Derivation of required sample size:** For desired power $(1-\beta)$ and significance $\alpha$:
$$n \geq \left(\frac{(z_{\alpha/2} + z_\beta) \cdot s_d}{\delta}\right)^2$$

where $\delta$ = minimum detectable effect size.

For example, to detect a 0.5 dB PSNR improvement with 95% confidence and 80% power, given $s_d = 1.0$:
$$n \geq \left(\frac{(1.96 + 0.84) \cdot 1.0}{0.5}\right)^2 = \left(\frac{2.8}{0.5}\right)^2 = 31.36 \implies n \geq 32$$

### Step 7: End-to-End Pipeline Integration
**Complete loss function:**
$$\mathcal{L}_{\text{total}} = \underbrace{\mathcal{L}_{\text{policy}}^{CLIP}}_{\text{PPO actor}} + \underbrace{c_1 \mathcal{L}_{\text{value}}}_{\text{critic MSE}} - \underbrace{c_2 H(\pi)}_{\text{entropy bonus}} + \underbrace{c_3 \|\theta\|_2^2}_{\text{weight decay}}$$

**Derivation of gradient magnitudes for balanced training:**
Each loss component should contribute approximately equally to $\|\nabla_\theta \mathcal{L}\|$:
$$\|\nabla \mathcal{L}_{\text{policy}}\| \approx c_1 \|\nabla \mathcal{L}_{\text{value}}\| \approx c_2 \|\nabla H\|$$

Monitor these during training and adjust $c_1, c_2$ accordingly. Typical starting values: $c_1 = 0.5$, $c_2 = 0.01$.

**Final model selection criterion (validation):**
$$\theta^* = \arg\max_\theta \frac{1}{|\mathcal{V}|}\sum_{I \in \mathcal{V}} \left[w_1 \text{PSNR}(\pi_\theta(I), I^{gt}) + w_2 \text{SSIM}(\pi_\theta(I), I^{gt})\right]$$

Select the checkpoint with the best validation score, NOT the lowest training loss (to avoid overfitting to the training distribution). $\blacksquare$

In [ ]:
# Download REAL open-source datasets for real-world projects
import torchvision
import subprocess
import sys

# MedMNIST for medical imaging (install + download)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'medmnist', '-q'])
import medmnist
from medmnist import ChestMNIST, DermaMNIST

# Chest X-rays (real medical images!)
chest_data = ChestMNIST(split='train', download=True, size=28)
print(f"✅ ChestMNIST: {len(chest_data)} real chest X-ray images")

# Dermatology images
derma_data = DermaMNIST(split='train', download=True, size=28)
print(f"✅ DermaMNIST: {len(derma_data)} real skin lesion images")

# EuroSAT for satellite imagery
try:
    eurosat = torchvision.datasets.EuroSAT(root='./data', download=True)
    print(f"✅ EuroSAT: {len(eurosat)} real satellite images (64x64, 10 land-use classes)")
except:
    print("⚠️ EuroSAT downloading...")

# CIFAR-10 for video/editing projects
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos for editing/video projects")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
from scipy import ndimage
from scipy.signal import convolve2d
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("\n\u2705 All dependencies loaded successfully!")

---

## 1. 🌟 Introduction - The Grand Vision

Throughout this course, we have built individual RL agents for different vision tasks:
- **Enhancement**: Adjusting brightness, contrast, denoising
- **Segmentation**: Partitioning images into meaningful regions
- **Detection**: Localizing objects with bounding boxes

Now we unify these into a **single pipeline** where each stage feeds the next,
creating a system greater than the sum of its parts.

### The Pipeline as a Multi-Stage MDP

We model the complete pipeline as a composition of Markov Decision Processes:

$$\mathcal{M}_{pipeline} = \mathcal{M}_1 \circ \mathcal{M}_2 \circ \mathcal{M}_3$$

where each $\mathcal{M}_k = (\mathcal{S}_k, \mathcal{A}_k, T_k, R_k, \gamma_k)$ represents a stage-specific MDP.

### Joint Optimization Objective

The pipeline is optimized to maximize the weighted sum of stage rewards:

$$J(\theta_1, \theta_2, \theta_3) = \mathbb{E}\left[\sum_{k=1}^{3} \alpha_k R_k(s_k, a_k)\right]$$

where $\alpha_k$ controls the relative importance of each stage.

### Inter-Stage State Transfer

The output of stage $k$ becomes the input of stage $k+1$:

$$s_{k+1} = g_k(s_k, \pi_k(s_k))$$

This creates a **differentiable chain** where upstream improvements cascade downstream.

### End-to-End Gradient Flow

For joint training, gradients flow through the entire pipeline:

$$\frac{\partial J}{\partial \theta_k} = \frac{\partial J}{\partial s_3} \cdot \frac{\partial s_3}{\partial s_2} \cdot \frac{\partial s_2}{\partial s_1} \cdot \frac{\partial s_1}{\partial \theta_k}$$

In practice, we use REINFORCE-style policy gradient with stage-specific baselines.

---

## 2. 🏗️ Pipeline Architecture

### Architectural Overview

```
┌─────────────┐    ┌───────────────┐    ┌────────────────┐    ┌─────────────┐
│ Degraded    │───▶│ Stage 1:      │───▶│ Stage 2:         │───▶│ Stage 3:    │
│ Input Image │    │ Enhancement   │    │ Segmentation     │    │ Detection   │
└─────────────┘    │ Agent (π₁)    │    │ Agent (π₂)       │    │ Agent (π₃) │
                   └───────────────┘    └────────────────┘    └─────────────┘
```

Each agent operates as a sequential decision maker:

| Stage | State | Actions | Reward |
|-------|-------|---------|--------|
| Enhancement | Current pixel statistics | Brightness, contrast, denoise, sharpen | PSNR/SSIM |
| Segmentation | Enhanced image features | Threshold, morph ops, region params | IoU |
| Detection | Segmentation map | Box position, size adjustments | Detection IoU |

In [ ]:
# Visualize the pipeline architecture
fig, ax = plt.subplots(1, 1, figsize=(16, 5))
ax.set_xlim(0, 16)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Complete RL Vision Pipeline Architecture', fontsize=16, fontweight='bold', pad=20)

# Draw pipeline stages
stages = [
    (1, 2.5, 'Degraded\nInput', '#ff6b6b'),
    (4.5, 2.5, 'Stage 1:\nEnhancement\nAgent (\u03c0\u2081)', '#4ecdc4'),
    (8, 2.5, 'Stage 2:\nSegmentation\nAgent (\u03c0\u2082)', '#45b7d1'),
    (11.5, 2.5, 'Stage 3:\nDetection\nAgent (\u03c0\u2083)', '#96ceb4'),
    (15, 2.5, 'Final\nOutput', '#feca57'),
]

for x, y, text, color in stages:
    bbox = dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.8, edgecolor='black', linewidth=2)
    ax.text(x, y, text, fontsize=11, ha='center', va='center', bbox=bbox, fontweight='bold')

# Draw arrows
arrow_props = dict(arrowstyle='->', lw=3, color='#2c3e50')
for i in range(4):
    x_start = stages[i][0] + 1.2
    x_end = stages[i+1][0] - 1.2
    ax.annotate('', xy=(x_end, 2.5), xytext=(x_start, 2.5), arrowprops=arrow_props)

# Draw reward signals
rewards = ['R\u2081: PSNR/SSIM', 'R\u2082: IoU', 'R\u2083: Det. IoU']
for i, (reward, stage) in enumerate(zip(rewards, stages[1:4])):
    ax.annotate(reward, xy=(stage[0], 1.0), fontsize=9, ha='center',
                color='#e74c3c', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='#fff3cd', alpha=0.8))
    ax.annotate('', xy=(stage[0], 1.4), xytext=(stage[0], 1.8),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#e74c3c', linestyle='dashed'))

plt.tight_layout()
plt.show()

---

## 3. 🎨 Synthetic Scene Generation

We generate complex synthetic scenes with multiple objects, then apply degradations
to create challenging inputs for our pipeline. Ground truth includes:
- Clean reference image
- Per-pixel segmentation mask
- Object bounding boxes

In [ ]:
class SyntheticSceneGenerator:
    """Generates synthetic scenes with geometric objects and full ground truth."""
    
    def __init__(self, image_size=64):
        self.image_size = image_size
        self.colors = [
            [1.0, 0.2, 0.2],  # Red
            [0.2, 0.8, 0.2],  # Green
            [0.2, 0.4, 1.0],  # Blue
            [1.0, 0.8, 0.0],  # Yellow
            [0.8, 0.2, 0.8],  # Purple
            [0.0, 0.8, 0.8],  # Cyan
        ]
    
    def _draw_circle(self, img, mask, cx, cy, radius, color, obj_id):
        y, x = np.ogrid[:self.image_size, :self.image_size]
        dist = np.sqrt((x - cx)**2 + (y - cy)**2)
        region = dist <= radius
        for c in range(3):
            img[:, :, c][region] = color[c]
        mask[region] = obj_id
        return max(0, cx - radius), max(0, cy - radius), min(self.image_size, cx + radius), min(self.image_size, cy + radius)
    
    def _draw_rectangle(self, img, mask, cx, cy, w, h, color, obj_id):
        x1 = max(0, int(cx - w//2))
        x2 = min(self.image_size, int(cx + w//2))
        y1 = max(0, int(cy - h//2))
        y2 = min(self.image_size, int(cy + h//2))
        for c in range(3):
            img[y1:y2, x1:x2, c] = color[c]
        mask[y1:y2, x1:x2] = obj_id
        return x1, y1, x2, y2
    
    def _draw_triangle(self, img, mask, cx, cy, size, color, obj_id):
        y, x = np.ogrid[:self.image_size, :self.image_size]
        # Equilateral triangle pointing up
        h = size * np.sqrt(3) / 2
        in_tri = ((y >= cy - h/2) & (y <= cy + h/2) &
                  (x >= cx - size/2 + (y - (cy - h/2)) * size / (2*h)) &
                  (x <= cx + size/2 - (y - (cy - h/2)) * size / (2*h)))
        for c in range(3):
            img[:, :, c][in_tri] = color[c]
        mask[in_tri] = obj_id
        x1 = max(0, int(cx - size//2))
        x2 = min(self.image_size, int(cx + size//2))
        y1 = max(0, int(cy - h//2))
        y2 = min(self.image_size, int(cy + h//2))
        return x1, y1, x2, y2
    
    def generate_scene(self, num_objects=None):
        if num_objects is None:
            num_objects = np.random.randint(3, 6)
        
        img = np.ones((self.image_size, self.image_size, 3)) * 0.15  # Dark background
        mask = np.zeros((self.image_size, self.image_size), dtype=np.int32)
        boxes = []
        
        for i in range(num_objects):
            shape_type = np.random.choice(['circle', 'rectangle', 'triangle'])
            color = self.colors[i % len(self.colors)]
            cx = np.random.randint(15, self.image_size - 15)
            cy = np.random.randint(15, self.image_size - 15)
            
            if shape_type == 'circle':
                radius = np.random.randint(5, 12)
                box = self._draw_circle(img, mask, cx, cy, radius, color, i+1)
            elif shape_type == 'rectangle':
                w = np.random.randint(8, 20)
                h = np.random.randint(8, 20)
                box = self._draw_rectangle(img, mask, cx, cy, w, h, color, i+1)
            else:
                size = np.random.randint(10, 20)
                box = self._draw_triangle(img, mask, cx, cy, size, color, i+1)
            
            boxes.append(box)
        
        img = np.clip(img, 0, 1)
        return img, mask, boxes
    
    def degrade_image(self, img, noise_level=0.15, blur_sigma=1.5, contrast_factor=0.5):
        """Apply degradations: noise, blur, low contrast."""
        degraded = img.copy()
        
        # Reduce contrast
        mean = degraded.mean()
        degraded = mean + contrast_factor * (degraded - mean)
        
        # Add Gaussian blur
        for c in range(3):
            degraded[:, :, c] = ndimage.gaussian_filter(degraded[:, :, c], sigma=blur_sigma)
        
        # Add noise
        noise = np.random.randn(*degraded.shape) * noise_level
        degraded = degraded + noise
        
        return np.clip(degraded, 0, 1)


# Create generator and generate sample scenes
generator = SyntheticSceneGenerator(image_size=64)

# Generate a batch of scenes
num_samples = 200
scenes = []
for _ in range(num_samples):
    clean, mask, boxes = generator.generate_scene()
    degraded = generator.degrade_image(clean)
    scenes.append({'clean': clean, 'degraded': degraded, 'mask': mask, 'boxes': boxes})

print(f"Generated {num_samples} synthetic scenes")
print(f"Image size: {scenes[0]['clean'].shape}")
print(f"Average objects per scene: {np.mean([len(s['boxes']) for s in scenes]):.1f}")

In [ ]:
# Visualize sample scenes with ground truth
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row in range(3):
    scene = scenes[row * 10]
    
    # Degraded input
    axes[row, 0].imshow(scene['degraded'])
    axes[row, 0].set_title('Degraded Input', fontsize=10)
    axes[row, 0].axis('off')
    
    # Clean reference
    axes[row, 1].imshow(scene['clean'])
    axes[row, 1].set_title('Clean (GT)', fontsize=10)
    axes[row, 1].axis('off')
    
    # Segmentation mask
    axes[row, 2].imshow(scene['mask'], cmap='tab10')
    axes[row, 2].set_title('Segmentation GT', fontsize=10)
    axes[row, 2].axis('off')
    
    # Bounding boxes
    axes[row, 3].imshow(scene['clean'])
    for box in scene['boxes']:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                            linewidth=2, edgecolor='lime', facecolor='none')
        axes[row, 3].add_patch(rect)
    axes[row, 3].set_title('Detection GT (Boxes)', fontsize=10)
    axes[row, 3].axis('off')

plt.suptitle('Synthetic Scenes with Full Ground Truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4. ✨ Stage 1: Enhancement Agent

The Enhancement Agent learns to restore degraded images by applying a sequence of
image processing operations.

### MDP Formulation

- **State**: Image statistics (mean, std, histogram features per channel)
- **Actions**: {brightness+, brightness-, contrast+, contrast-, denoise, sharpen}
- **Reward**: Improvement in PSNR/SSIM relative to ground truth:

$$R_1 = \Delta\text{PSNR} + \lambda \cdot \Delta\text{SSIM}$$

where $\Delta$ denotes the change from the previous step.

In [ ]:
class EnhancementEnvironment:
    """RL Environment for image enhancement."""
    
    def __init__(self, max_steps=10):
        self.max_steps = max_steps
        self.action_names = ['brightness+', 'brightness-', 'contrast+', 
                            'contrast-', 'denoise', 'sharpen']
        self.n_actions = len(self.action_names)
        self.state_dim = 18  # 6 features per channel
    
    def reset(self, scene):
        self.clean = scene['clean']
        self.current = scene['degraded'].copy()
        self.step_count = 0
        self.prev_quality = self._compute_quality()
        return self._get_state()
    
    def _get_state(self):
        features = []
        for c in range(3):
            ch = self.current[:, :, c]
            features.extend([
                ch.mean(), ch.std(), np.median(ch),
                ch.min(), ch.max(), np.percentile(ch, 75) - np.percentile(ch, 25)
            ])
        return np.array(features, dtype=np.float32)
    
    def _compute_quality(self):
        p = psnr(self.clean, self.current, data_range=1.0)
        s = ssim(self.clean, self.current, data_range=1.0, channel_axis=2)
        return p + 10 * s
    
    def step(self, action):
        self.step_count += 1
        
        if action == 0:  # brightness+
            self.current = np.clip(self.current + 0.03, 0, 1)
        elif action == 1:  # brightness-
            self.current = np.clip(self.current - 0.03, 0, 1)
        elif action == 2:  # contrast+
            mean = self.current.mean()
            self.current = np.clip(mean + 1.15 * (self.current - mean), 0, 1)
        elif action == 3:  # contrast-
            mean = self.current.mean()
            self.current = np.clip(mean + 0.9 * (self.current - mean), 0, 1)
        elif action == 4:  # denoise
            for c in range(3):
                self.current[:, :, c] = ndimage.median_filter(self.current[:, :, c], size=3)
        elif action == 5:  # sharpen
            kernel = np.array([[0, -0.5, 0], [-0.5, 3, -0.5], [0, -0.5, 0]])
            for c in range(3):
                self.current[:, :, c] = np.clip(
                    convolve2d(self.current[:, :, c], kernel, mode='same', boundary='symm'), 0, 1)
        
        quality = self._compute_quality()
        reward = (quality - self.prev_quality) * 0.1
        self.prev_quality = quality
        
        done = self.step_count >= self.max_steps
        return self._get_state(), reward, done
    
    def get_enhanced_image(self):
        return self.current.copy()


print("Enhancement Environment:")
print(f"  State dimension: 18 (6 statistics x 3 channels)")
print(f"  Action space: 6 actions")
print(f"  Max steps per episode: 10")

In [ ]:
class PolicyNetwork(nn.Module):
    """Shared policy network architecture for all agents."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
        self.value_head = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        logits = self.network(x)
        value = self.value_head(x)
        return logits, value
    
    def get_action(self, state, deterministic=False):
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            logits, value = self.forward(state_t)
            probs = F.softmax(logits, dim=-1)
            if deterministic:
                action = probs.argmax(dim=-1).item()
            else:
                action = torch.multinomial(probs, 1).item()
        return action, probs.squeeze().cpu().numpy(), value.item()


class REINFORCETrainer:
    """REINFORCE with baseline for training RL agents."""
    
    def __init__(self, policy, lr=3e-4, gamma=0.99):
        self.policy = policy
        self.optimizer = optim.Adam(policy.parameters(), lr=lr)
        self.gamma = gamma
    
    def train_episode(self, env, scene):
        state = env.reset(scene)
        
        log_probs = []
        values = []
        rewards = []
        
        done = False
        while not done:
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            logits, value = self.policy(state_t)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            
            next_state, reward, done = env.step(action.item())
            
            log_probs.append(dist.log_prob(action))
            values.append(value.squeeze())
            rewards.append(reward)
            state = next_state
        
        # Compute returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)
        
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # Policy and value loss
        policy_loss = 0
        value_loss = 0
        for log_prob, value, G in zip(log_probs, values, returns):
            advantage = G - value.detach()
            policy_loss -= log_prob * advantage
            value_loss += F.mse_loss(value, G)
        
        loss = policy_loss + 0.5 * value_loss
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
        self.optimizer.step()
        
        return sum(rewards)


print("\u2705 Policy Network and REINFORCE Trainer defined")

In [ ]:
# Train Enhancement Agent
print("Training Stage 1: Enhancement Agent...")
print("="*50)

enhance_env = EnhancementEnvironment(max_steps=10)
enhance_policy = PolicyNetwork(state_dim=18, action_dim=6, hidden_dim=128).to(device)
enhance_trainer = REINFORCETrainer(enhance_policy, lr=3e-4)

enhance_rewards = []
enhance_psnr_history = []
n_episodes = 300

for episode in range(n_episodes):
    scene = scenes[np.random.randint(len(scenes))]
    ep_reward = enhance_trainer.train_episode(enhance_env, scene)
    enhance_rewards.append(ep_reward)
    
    # Track PSNR
    if episode % 10 == 0:
        test_scene = scenes[0]
        enhance_env.reset(test_scene)
        state = enhance_env._get_state()
        for _ in range(10):
            action, _, _ = enhance_policy.get_action(state, deterministic=True)
            state, _, done = enhance_env.step(action)
            if done:
                break
        enhanced = enhance_env.get_enhanced_image()
        current_psnr = psnr(test_scene['clean'], enhanced, data_range=1.0)
        enhance_psnr_history.append(current_psnr)
    
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(enhance_rewards[-50:])
        print(f"  Episode {episode+1}/{n_episodes} | Avg Reward: {avg_reward:.4f} | PSNR: {current_psnr:.2f} dB")

print("\n\u2705 Enhancement Agent training complete!")

In [ ]:
# Demonstrate Enhancement Agent
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row in range(2):
    scene = scenes[row * 5 + 1]
    
    # Run enhancement
    state = enhance_env.reset(scene)
    actions_taken = []
    for _ in range(10):
        action, _, _ = enhance_policy.get_action(state, deterministic=True)
        actions_taken.append(enhance_env.action_names[action])
        state, _, done = enhance_env.step(action)
        if done:
            break
    enhanced = enhance_env.get_enhanced_image()
    
    # Compute metrics
    psnr_before = psnr(scene['clean'], scene['degraded'], data_range=1.0)
    psnr_after = psnr(scene['clean'], enhanced, data_range=1.0)
    ssim_before = ssim(scene['clean'], scene['degraded'], data_range=1.0, channel_axis=2)
    ssim_after = ssim(scene['clean'], enhanced, data_range=1.0, channel_axis=2)
    
    axes[row, 0].imshow(scene['degraded'])
    axes[row, 0].set_title(f'Degraded\nPSNR: {psnr_before:.1f} dB', fontsize=10)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(enhanced)
    axes[row, 1].set_title(f'Enhanced (RL)\nPSNR: {psnr_after:.1f} dB', fontsize=10)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(scene['clean'])
    axes[row, 2].set_title(f'Ground Truth', fontsize=10)
    axes[row, 2].axis('off')
    
    # Show improvement
    diff = np.abs(scene['clean'] - enhanced).mean(axis=2)
    axes[row, 3].imshow(diff, cmap='hot', vmin=0, vmax=0.3)
    axes[row, 3].set_title(f'Error Map\n\u0394PSNR: +{psnr_after-psnr_before:.1f} dB', fontsize=10)
    axes[row, 3].axis('off')

plt.suptitle('Stage 1: Enhancement Agent Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 5. 🧩 Stage 2: Segmentation Agent

The Segmentation Agent processes the enhanced image to produce a semantic segmentation map.

### MDP Formulation

- **State**: Enhanced image statistics + current segmentation quality features
- **Actions**: Threshold adjustment, morphological operations, region parameters
- **Reward**: Intersection over Union (IoU) with ground truth:

$$IoU = \frac{|\text{Pred} \cap \text{GT}|}{|\text{Pred} \cup \text{GT}|}$$

The overall segmentation reward accounts for multi-class accuracy:

$$R_2 = \frac{1}{K}\sum_{k=1}^{K} IoU_k$$

where $K$ is the number of object classes.

In [ ]:
class SegmentationEnvironment:
    """RL Environment for image segmentation."""
    
    def __init__(self, max_steps=8):
        self.max_steps = max_steps
        self.action_names = [
            'threshold_up', 'threshold_down',
            'dilate', 'erode',
            'open', 'close',
            'adaptive_threshold', 'color_segment'
        ]
        self.n_actions = len(self.action_names)
        self.state_dim = 20
    
    def reset(self, enhanced_img, gt_mask):
        self.image = enhanced_img
        self.gt_mask = (gt_mask > 0).astype(np.float32)  # Binary: object vs background
        self.threshold = 0.4
        self.step_count = 0
        
        # Initialize segmentation via simple thresholding
        gray = np.mean(self.image, axis=2)
        self.current_seg = (gray > self.threshold).astype(np.float32)
        self.prev_iou = self._compute_iou()
        return self._get_state()
    
    def _get_state(self):
        gray = np.mean(self.image, axis=2)
        features = [
            gray.mean(), gray.std(),
            self.current_seg.mean(), self.current_seg.std(),
            self.threshold,
            self._compute_iou(),
            np.percentile(gray, 25), np.percentile(gray, 50), np.percentile(gray, 75),
            self.image[:,:,0].mean(), self.image[:,:,1].mean(), self.image[:,:,2].mean(),
            self.image[:,:,0].std(), self.image[:,:,1].std(), self.image[:,:,2].std(),
            ndimage.label(self.current_seg)[1] / 10.0,  # Number of connected components
            self.current_seg.sum() / self.current_seg.size,  # Foreground ratio
            self.step_count / self.max_steps,
            float(self.prev_iou),
            float(np.abs(gray - self.threshold).mean()),
        ]
        return np.array(features, dtype=np.float32)
    
    def _compute_iou(self):
        pred = self.current_seg > 0.5
        gt = self.gt_mask > 0.5
        intersection = np.logical_and(pred, gt).sum()
        union = np.logical_or(pred, gt).sum()
        return intersection / (union + 1e-8)
    
    def step(self, action):
        self.step_count += 1
        gray = np.mean(self.image, axis=2)
        
        if action == 0:  # threshold_up
            self.threshold = min(0.9, self.threshold + 0.05)
            self.current_seg = (gray > self.threshold).astype(np.float32)
        elif action == 1:  # threshold_down
            self.threshold = max(0.1, self.threshold - 0.05)
            self.current_seg = (gray > self.threshold).astype(np.float32)
        elif action == 2:  # dilate
            self.current_seg = ndimage.binary_dilation(self.current_seg, iterations=1).astype(np.float32)
        elif action == 3:  # erode
            self.current_seg = ndimage.binary_erosion(self.current_seg, iterations=1).astype(np.float32)
        elif action == 4:  # open
            self.current_seg = ndimage.binary_opening(self.current_seg, iterations=1).astype(np.float32)
        elif action == 5:  # close
            self.current_seg = ndimage.binary_closing(self.current_seg, iterations=1).astype(np.float32)
        elif action == 6:  # adaptive_threshold
            local_mean = ndimage.uniform_filter(gray, size=7)
            self.current_seg = (gray > local_mean - 0.05).astype(np.float32)
        elif action == 7:  # color_segment
            color_dist = np.sqrt(np.sum((self.image - 0.15)**2, axis=2))  # Distance from background
            self.current_seg = (color_dist > 0.3).astype(np.float32)
        
        iou = self._compute_iou()
        reward = (iou - self.prev_iou) * 5.0  # Scale reward
        self.prev_iou = iou
        
        done = self.step_count >= self.max_steps
        return self._get_state(), reward, done
    
    def get_segmentation(self):
        return self.current_seg.copy()


print("Segmentation Environment:")
print(f"  State dimension: 20")
print(f"  Action space: 8 actions")
print(f"  Max steps per episode: 8")

In [ ]:
# Train Segmentation Agent
print("Training Stage 2: Segmentation Agent...")
print("="*50)

seg_env = SegmentationEnvironment(max_steps=8)
seg_policy = PolicyNetwork(state_dim=20, action_dim=8, hidden_dim=128).to(device)
seg_trainer = REINFORCETrainer(seg_policy, lr=3e-4)

seg_rewards = []
seg_iou_history = []
n_episodes = 300

for episode in range(n_episodes):
    scene = scenes[np.random.randint(len(scenes))]
    
    # First enhance the image (use trained enhancement agent)
    state = enhance_env.reset(scene)
    for _ in range(10):
        action, _, _ = enhance_policy.get_action(state, deterministic=True)
        state, _, done = enhance_env.step(action)
        if done:
            break
    enhanced = enhance_env.get_enhanced_image()
    
    # Train segmentation on enhanced image
    seg_state = seg_env.reset(enhanced, scene['mask'])
    ep_reward = 0
    done = False
    
    log_probs = []
    values = []
    rewards_list = []
    
    while not done:
        state_t = torch.FloatTensor(seg_state).unsqueeze(0).to(device)
        logits, value = seg_policy(state_t)
        probs = F.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        
        seg_state, reward, done = seg_env.step(action.item())
        
        log_probs.append(dist.log_prob(action))
        values.append(value.squeeze())
        rewards_list.append(reward)
        ep_reward += reward
    
    # Update policy
    returns = []
    G = 0
    for r in reversed(rewards_list):
        G = r + 0.99 * G
        returns.insert(0, G)
    returns = torch.FloatTensor(returns).to(device)
    if len(returns) > 1:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    
    policy_loss = 0
    value_loss = 0
    for lp, v, G in zip(log_probs, values, returns):
        advantage = G - v.detach()
        policy_loss -= lp * advantage
        value_loss += F.mse_loss(v, G)
    
    loss = policy_loss + 0.5 * value_loss
    seg_trainer.optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(seg_policy.parameters(), 0.5)
    seg_trainer.optimizer.step()
    
    seg_rewards.append(ep_reward)
    
    # Track IoU
    if episode % 10 == 0:
        test_iou = seg_env._compute_iou()
        seg_iou_history.append(test_iou)
    
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(seg_rewards[-50:])
        print(f"  Episode {episode+1}/{n_episodes} | Avg Reward: {avg_reward:.4f} | IoU: {test_iou:.4f}")

print("\n\u2705 Segmentation Agent training complete!")

In [ ]:
# Demonstrate Segmentation Agent
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row in range(2):
    scene = scenes[row * 7 + 2]
    
    # Enhance first
    state = enhance_env.reset(scene)
    for _ in range(10):
        action, _, _ = enhance_policy.get_action(state, deterministic=True)
        state, _, done = enhance_env.step(action)
        if done:
            break
    enhanced = enhance_env.get_enhanced_image()
    
    # Then segment
    seg_state = seg_env.reset(enhanced, scene['mask'])
    for _ in range(8):
        action, _, _ = seg_policy.get_action(seg_state, deterministic=True)
        seg_state, _, done = seg_env.step(action)
        if done:
            break
    segmentation = seg_env.get_segmentation()
    final_iou = seg_env._compute_iou()
    
    axes[row, 0].imshow(scene['degraded'])
    axes[row, 0].set_title('Original (Degraded)', fontsize=10)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(enhanced)
    axes[row, 1].set_title('After Enhancement', fontsize=10)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(segmentation, cmap='gray')
    axes[row, 2].set_title(f'RL Segmentation\nIoU: {final_iou:.3f}', fontsize=10)
    axes[row, 2].axis('off')
    
    axes[row, 3].imshow(scene['mask'] > 0, cmap='gray')
    axes[row, 3].set_title('Ground Truth Mask', fontsize=10)
    axes[row, 3].axis('off')

plt.suptitle('Stage 2: Segmentation Agent Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 6. 🔍 Stage 3: Detection Agent

The Detection Agent places and refines bounding boxes around detected objects.

### MDP Formulation

- **State**: Segmentation features + current box parameters
- **Actions**: Move box (up/down/left/right), resize (grow/shrink), confirm
- **Reward**: IoU with ground truth bounding boxes:

$$R_3 = \max_{b \in GT} \; IoU(\hat{b}, b)$$

where $\hat{b}$ is the predicted box and $b$ ranges over ground truth boxes.

The detection IoU for axis-aligned boxes:

$$IoU_{det} = \frac{\text{Area}(\hat{b} \cap b)}{\text{Area}(\hat{b} \cup b)} = \frac{\text{Area}(\hat{b} \cap b)}{\text{Area}(\hat{b}) + \text{Area}(b) - \text{Area}(\hat{b} \cap b)}$$

In [ ]:
class DetectionEnvironment:
    """RL Environment for object detection via bounding box refinement."""
    
    def __init__(self, image_size=64, max_steps=12):
        self.image_size = image_size
        self.max_steps = max_steps
        self.action_names = [
            'move_right', 'move_left', 'move_down', 'move_up',
            'grow_w', 'shrink_w', 'grow_h', 'shrink_h',
            'grow_all', 'shrink_all'
        ]
        self.n_actions = len(self.action_names)
        self.state_dim = 16
        self.move_step = 2
        self.resize_step = 2
    
    def reset(self, segmentation, gt_boxes):
        self.segmentation = segmentation
        self.gt_boxes = gt_boxes
        self.step_count = 0
        
        # Initialize box from segmentation centroid
        if segmentation.sum() > 0:
            coords = np.where(segmentation > 0.5)
            cy, cx = coords[0].mean(), coords[1].mean()
            h_spread = coords[0].std() * 2
            w_spread = coords[1].std() * 2
        else:
            cx, cy = self.image_size // 2, self.image_size // 2
            w_spread, h_spread = 15, 15
        
        # Current predicted box [x1, y1, x2, y2]
        self.box = [
            max(0, int(cx - w_spread)),
            max(0, int(cy - h_spread)),
            min(self.image_size, int(cx + w_spread)),
            min(self.image_size, int(cy + h_spread))
        ]
        
        self.prev_iou = self._compute_best_iou()
        return self._get_state()
    
    def _get_state(self):
        x1, y1, x2, y2 = self.box
        w = x2 - x1
        h = y2 - y1
        cx = (x1 + x2) / 2 / self.image_size
        cy = (y1 + y2) / 2 / self.image_size
        
        # Segmentation features inside and around box
        box_region = self.segmentation[y1:y2, x1:x2]
        inside_density = box_region.mean() if box_region.size > 0 else 0
        
        # Features around box edges
        pad = 3
        left_region = self.segmentation[y1:y2, max(0,x1-pad):x1]
        right_region = self.segmentation[y1:y2, x2:min(self.image_size,x2+pad)]
        top_region = self.segmentation[max(0,y1-pad):y1, x1:x2]
        bottom_region = self.segmentation[y2:min(self.image_size,y2+pad), x1:x2]
        
        features = [
            cx, cy, w / self.image_size, h / self.image_size,
            inside_density,
            left_region.mean() if left_region.size > 0 else 0,
            right_region.mean() if right_region.size > 0 else 0,
            top_region.mean() if top_region.size > 0 else 0,
            bottom_region.mean() if bottom_region.size > 0 else 0,
            self.segmentation.mean(),
            float(self.prev_iou),
            self.step_count / self.max_steps,
            float(x1) / self.image_size,
            float(y1) / self.image_size,
            float(x2) / self.image_size,
            float(y2) / self.image_size,
        ]
        return np.array(features, dtype=np.float32)
    
    def _compute_box_iou(self, box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - inter
        
        return inter / (union + 1e-8)
    
    def _compute_best_iou(self):
        if not self.gt_boxes:
            return 0.0
        return max(self._compute_box_iou(self.box, gt) for gt in self.gt_boxes)
    
    def step(self, action):
        self.step_count += 1
        x1, y1, x2, y2 = self.box
        
        if action == 0:  # move_right
            x1 += self.move_step
            x2 += self.move_step
        elif action == 1:  # move_left
            x1 -= self.move_step
            x2 -= self.move_step
        elif action == 2:  # move_down
            y1 += self.move_step
            y2 += self.move_step
        elif action == 3:  # move_up
            y1 -= self.move_step
            y2 -= self.move_step
        elif action == 4:  # grow_w
            x1 -= self.resize_step
            x2 += self.resize_step
        elif action == 5:  # shrink_w
            x1 += self.resize_step
            x2 -= self.resize_step
        elif action == 6:  # grow_h
            y1 -= self.resize_step
            y2 += self.resize_step
        elif action == 7:  # shrink_h
            y1 += self.resize_step
            y2 -= self.resize_step
        elif action == 8:  # grow_all
            x1 -= self.resize_step
            y1 -= self.resize_step
            x2 += self.resize_step
            y2 += self.resize_step
        elif action == 9:  # shrink_all
            x1 += self.resize_step
            y1 += self.resize_step
            x2 -= self.resize_step
            y2 -= self.resize_step
        
        # Clamp and ensure valid box
        x1 = max(0, min(self.image_size - 4, x1))
        y1 = max(0, min(self.image_size - 4, y1))
        x2 = max(x1 + 4, min(self.image_size, x2))
        y2 = max(y1 + 4, min(self.image_size, y2))
        
        self.box = [x1, y1, x2, y2]
        
        iou = self._compute_best_iou()
        reward = (iou - self.prev_iou) * 10.0
        self.prev_iou = iou
        
        done = self.step_count >= self.max_steps
        return self._get_state(), reward, done
    
    def get_detection(self):
        return list(self.box)


print("Detection Environment:")
print(f"  State dimension: 16")
print(f"  Action space: 10 actions")
print(f"  Max steps per episode: 12")

In [ ]:
# Train Detection Agent
print("Training Stage 3: Detection Agent...")
print("="*50)

det_env = DetectionEnvironment(image_size=64, max_steps=12)
det_policy = PolicyNetwork(state_dim=16, action_dim=10, hidden_dim=128).to(device)
det_trainer = REINFORCETrainer(det_policy, lr=3e-4)

det_rewards = []
det_iou_history = []
n_episodes = 300

for episode in range(n_episodes):
    scene = scenes[np.random.randint(len(scenes))]
    
    # Run through enhancement
    state = enhance_env.reset(scene)
    for _ in range(10):
        action, _, _ = enhance_policy.get_action(state, deterministic=True)
        state, _, done = enhance_env.step(action)
        if done:
            break
    enhanced = enhance_env.get_enhanced_image()
    
    # Run through segmentation
    seg_state = seg_env.reset(enhanced, scene['mask'])
    for _ in range(8):
        action, _, _ = seg_policy.get_action(seg_state, deterministic=True)
        seg_state, _, done = seg_env.step(action)
        if done:
            break
    segmentation = seg_env.get_segmentation()
    
    # Train detection agent
    det_state = det_env.reset(segmentation, scene['boxes'])
    ep_reward = 0
    done = False
    
    log_probs = []
    values = []
    rewards_list = []
    
    while not done:
        state_t = torch.FloatTensor(det_state).unsqueeze(0).to(device)
        logits, value = det_policy(state_t)
        probs = F.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        
        det_state, reward, done = det_env.step(action.item())
        
        log_probs.append(dist.log_prob(action))
        values.append(value.squeeze())
        rewards_list.append(reward)
        ep_reward += reward
    
    # Update policy
    returns = []
    G = 0
    for r in reversed(rewards_list):
        G = r + 0.99 * G
        returns.insert(0, G)
    returns = torch.FloatTensor(returns).to(device)
    if len(returns) > 1:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    
    policy_loss = 0
    value_loss = 0
    for lp, v, G in zip(log_probs, values, returns):
        advantage = G - v.detach()
        policy_loss -= lp * advantage
        value_loss += F.mse_loss(v, G)
    
    loss = policy_loss + 0.5 * value_loss
    det_trainer.optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(det_policy.parameters(), 0.5)
    det_trainer.optimizer.step()
    
    det_rewards.append(ep_reward)
    
    # Track detection IoU
    if episode % 10 == 0:
        test_iou = det_env._compute_best_iou()
        det_iou_history.append(test_iou)
    
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(det_rewards[-50:])
        print(f"  Episode {episode+1}/{n_episodes} | Avg Reward: {avg_reward:.4f} | Det IoU: {test_iou:.4f}")

print("\n\u2705 Detection Agent training complete!")

In [ ]:
# Demonstrate Detection Agent
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for row in range(2):
    scene = scenes[row * 12 + 3]
    
    # Full pipeline
    state = enhance_env.reset(scene)
    for _ in range(10):
        action, _, _ = enhance_policy.get_action(state, deterministic=True)
        state, _, done = enhance_env.step(action)
        if done:
            break
    enhanced = enhance_env.get_enhanced_image()
    
    seg_state = seg_env.reset(enhanced, scene['mask'])
    for _ in range(8):
        action, _, _ = seg_policy.get_action(seg_state, deterministic=True)
        seg_state, _, done = seg_env.step(action)
        if done:
            break
    segmentation = seg_env.get_segmentation()
    
    det_state = det_env.reset(segmentation, scene['boxes'])
    for _ in range(12):
        action, _, _ = det_policy.get_action(det_state, deterministic=True)
        det_state, _, done = det_env.step(action)
        if done:
            break
    pred_box = det_env.get_detection()
    det_iou = det_env._compute_best_iou()
    
    # Visualize
    axes[row, 0].imshow(segmentation, cmap='gray')
    axes[row, 0].set_title('Segmentation Input', fontsize=10)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(enhanced)
    rect = plt.Rectangle((pred_box[0], pred_box[1]), pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
                         linewidth=2, edgecolor='red', facecolor='none', linestyle='-')
    axes[row, 1].add_patch(rect)
    axes[row, 1].set_title(f'Predicted Box (IoU: {det_iou:.3f})', fontsize=10)
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(scene['clean'])
    for gt_box in scene['boxes']:
        rect_gt = plt.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                              linewidth=2, edgecolor='lime', facecolor='none')
        axes[row, 2].add_patch(rect_gt)
    rect_pred = plt.Rectangle((pred_box[0], pred_box[1]), pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
                             linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    axes[row, 2].add_patch(rect_pred)
    axes[row, 2].legend([mpatches.Patch(edgecolor='lime', facecolor='none', linewidth=2),
                         mpatches.Patch(edgecolor='red', facecolor='none', linewidth=2, linestyle='--')],
                        ['GT Boxes', 'Predicted'], loc='lower right', fontsize=8)
    axes[row, 2].set_title('GT vs Predicted', fontsize=10)
    axes[row, 2].axis('off')

plt.suptitle('Stage 3: Detection Agent Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 7. 🔗 End-to-End Pipeline

Now we chain all three agents together into the complete pipeline and evaluate
the end-to-end performance.

### Pipeline Execution

For each input image $I$:

$$I \xrightarrow{\pi_1} I_{enhanced} \xrightarrow{\pi_2} M_{seg} \xrightarrow{\pi_3} B_{det}$$

The pipeline quality is measured by:

$$Q_{pipeline} = \alpha_1 \cdot \text{PSNR}(I_{enh}, I_{clean}) + \alpha_2 \cdot \text{IoU}_{seg} + \alpha_3 \cdot \text{IoU}_{det}$$

In [ ]:
class CompletePipeline:
    """End-to-end RL Vision Pipeline combining all three stages."""
    
    def __init__(self, enhance_env, enhance_policy, seg_env, seg_policy, det_env, det_policy):
        self.enhance_env = enhance_env
        self.enhance_policy = enhance_policy
        self.seg_env = seg_env
        self.seg_policy = seg_policy
        self.det_env = det_env
        self.det_policy = det_policy
    
    def run(self, scene):
        """Run the complete pipeline on a scene."""
        results = {'input': scene['degraded'].copy()}
        
        # Stage 1: Enhancement
        state = self.enhance_env.reset(scene)
        for _ in range(10):
            action, _, _ = self.enhance_policy.get_action(state, deterministic=True)
            state, _, done = self.enhance_env.step(action)
            if done:
                break
        results['enhanced'] = self.enhance_env.get_enhanced_image()
        results['psnr'] = psnr(scene['clean'], results['enhanced'], data_range=1.0)
        results['ssim'] = ssim(scene['clean'], results['enhanced'], data_range=1.0, channel_axis=2)
        
        # Stage 2: Segmentation
        seg_state = self.seg_env.reset(results['enhanced'], scene['mask'])
        for _ in range(8):
            action, _, _ = self.seg_policy.get_action(seg_state, deterministic=True)
            seg_state, _, done = self.seg_env.step(action)
            if done:
                break
        results['segmentation'] = self.seg_env.get_segmentation()
        results['seg_iou'] = self.seg_env._compute_iou()
        
        # Stage 3: Detection
        det_state = self.det_env.reset(results['segmentation'], scene['boxes'])
        for _ in range(12):
            action, _, _ = self.det_policy.get_action(det_state, deterministic=True)
            det_state, _, done = self.det_env.step(action)
            if done:
                break
        results['detection'] = self.det_env.get_detection()
        results['det_iou'] = self.det_env._compute_best_iou()
        
        return results


# Create the pipeline
pipeline = CompletePipeline(
    enhance_env, enhance_policy,
    seg_env, seg_policy,
    det_env, det_policy
)

print("\u2705 Complete RL Vision Pipeline assembled!")
print("\nPipeline stages:")
print("  1. Enhancement Agent (18-dim state, 6 actions, 10 steps)")
print("  2. Segmentation Agent (20-dim state, 8 actions, 8 steps)")
print("  3. Detection Agent (16-dim state, 10 actions, 12 steps)")

In [ ]:
# Run full pipeline on test scenes and visualize
fig = plt.figure(figsize=(18, 14))
gs = GridSpec(3, 5, figure=fig, hspace=0.4, wspace=0.3)

test_indices = [50, 75, 100]

for row, idx in enumerate(test_indices):
    scene = scenes[idx]
    results = pipeline.run(scene)
    
    # Original degraded
    ax = fig.add_subplot(gs[row, 0])
    ax.imshow(results['input'])
    ax.set_title('Input\n(Degraded)', fontsize=9)
    ax.axis('off')
    
    # Enhanced
    ax = fig.add_subplot(gs[row, 1])
    ax.imshow(results['enhanced'])
    ax.set_title(f'Stage 1: Enhanced\nPSNR: {results["psnr"]:.1f} dB', fontsize=9)
    ax.axis('off')
    
    # Segmented
    ax = fig.add_subplot(gs[row, 2])
    ax.imshow(results['segmentation'], cmap='gray')
    ax.set_title(f'Stage 2: Segmented\nIoU: {results["seg_iou"]:.3f}', fontsize=9)
    ax.axis('off')
    
    # Detection
    ax = fig.add_subplot(gs[row, 3])
    ax.imshow(results['enhanced'])
    box = results['detection']
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                         linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(f'Stage 3: Detected\nIoU: {results["det_iou"]:.3f}', fontsize=9)
    ax.axis('off')
    
    # Ground truth comparison
    ax = fig.add_subplot(gs[row, 4])
    ax.imshow(scene['clean'])
    for gt_box in scene['boxes']:
        rect_gt = plt.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                              linewidth=1.5, edgecolor='lime', facecolor='none')
        ax.add_patch(rect_gt)
    ax.set_title('Ground Truth', fontsize=9)
    ax.axis('off')

fig.suptitle('Complete End-to-End RL Vision Pipeline', fontsize=15, fontweight='bold')
plt.show()

---

## 8. 📊 Comprehensive Evaluation

We perform a thorough evaluation including:
- Per-stage training curves
- Pipeline performance metrics
- Ablation studies
- Error analysis

In [ ]:
# Per-stage training curves
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Enhancement reward curve
window = 20
enhance_smooth = np.convolve(enhance_rewards, np.ones(window)/window, mode='valid')
axes[0, 0].plot(enhance_smooth, color='#4ecdc4', linewidth=2)
axes[0, 0].fill_between(range(len(enhance_smooth)), enhance_smooth, alpha=0.3, color='#4ecdc4')
axes[0, 0].set_title('Stage 1: Enhancement Reward', fontweight='bold')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Episode Reward')
axes[0, 0].grid(True, alpha=0.3)

# Segmentation reward curve
seg_smooth = np.convolve(seg_rewards, np.ones(window)/window, mode='valid')
axes[0, 1].plot(seg_smooth, color='#45b7d1', linewidth=2)
axes[0, 1].fill_between(range(len(seg_smooth)), seg_smooth, alpha=0.3, color='#45b7d1')
axes[0, 1].set_title('Stage 2: Segmentation Reward', fontweight='bold')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Episode Reward')
axes[0, 1].grid(True, alpha=0.3)

# Detection reward curve
det_smooth = np.convolve(det_rewards, np.ones(window)/window, mode='valid')
axes[0, 2].plot(det_smooth, color='#96ceb4', linewidth=2)
axes[0, 2].fill_between(range(len(det_smooth)), det_smooth, alpha=0.3, color='#96ceb4')
axes[0, 2].set_title('Stage 3: Detection Reward', fontweight='bold')
axes[0, 2].set_xlabel('Episode')
axes[0, 2].set_ylabel('Episode Reward')
axes[0, 2].grid(True, alpha=0.3)

# PSNR improvement over training
axes[1, 0].plot(enhance_psnr_history, 'o-', color='#4ecdc4', markersize=3, linewidth=1.5)
axes[1, 0].axhline(y=enhance_psnr_history[0], color='gray', linestyle='--', alpha=0.7, label='Initial')
axes[1, 0].set_title('Enhancement: PSNR Over Training', fontweight='bold')
axes[1, 0].set_xlabel('Evaluation Step')
axes[1, 0].set_ylabel('PSNR (dB)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Segmentation IoU over training
axes[1, 1].plot(seg_iou_history, 'o-', color='#45b7d1', markersize=3, linewidth=1.5)
axes[1, 1].axhline(y=seg_iou_history[0], color='gray', linestyle='--', alpha=0.7, label='Initial')
axes[1, 1].set_title('Segmentation: IoU Over Training', fontweight='bold')
axes[1, 1].set_xlabel('Evaluation Step')
axes[1, 1].set_ylabel('IoU')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Detection IoU over training
axes[1, 2].plot(det_iou_history, 'o-', color='#96ceb4', markersize=3, linewidth=1.5)
axes[1, 2].axhline(y=det_iou_history[0], color='gray', linestyle='--', alpha=0.7, label='Initial')
axes[1, 2].set_title('Detection: IoU Over Training', fontweight='bold')
axes[1, 2].set_xlabel('Evaluation Step')
axes[1, 2].set_ylabel('Detection IoU')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Training Progress Across All Pipeline Stages', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive pipeline evaluation on test set
print("Running comprehensive pipeline evaluation...")
print("="*60)

n_test = 50
test_scenes = scenes[150:150+n_test]

metrics = {
    'psnr_before': [], 'psnr_after': [],
    'ssim_before': [], 'ssim_after': [],
    'seg_iou': [], 'det_iou': [],
    'pipeline_time_steps': []
}

for scene in test_scenes:
    # Before enhancement
    metrics['psnr_before'].append(psnr(scene['clean'], scene['degraded'], data_range=1.0))
    metrics['ssim_before'].append(ssim(scene['clean'], scene['degraded'], data_range=1.0, channel_axis=2))
    
    # Run pipeline
    results = pipeline.run(scene)
    
    metrics['psnr_after'].append(results['psnr'])
    metrics['ssim_after'].append(results['ssim'])
    metrics['seg_iou'].append(results['seg_iou'])
    metrics['det_iou'].append(results['det_iou'])
    metrics['pipeline_time_steps'].append(10 + 8 + 12)  # Total steps

print(f"\nPipeline Performance Summary (n={n_test} test images):")
print(f"-" * 60)
print(f"Stage 1 - Enhancement:")
print(f"  PSNR: {np.mean(metrics['psnr_before']):.2f} \u2192 {np.mean(metrics['psnr_after']):.2f} dB (\u0394 = +{np.mean(np.array(metrics['psnr_after'])-np.array(metrics['psnr_before'])):.2f})")
print(f"  SSIM: {np.mean(metrics['ssim_before']):.4f} \u2192 {np.mean(metrics['ssim_after']):.4f}")
print(f"\nStage 2 - Segmentation:")
print(f"  Mean IoU: {np.mean(metrics['seg_iou']):.4f} \u00b1 {np.std(metrics['seg_iou']):.4f}")
print(f"\nStage 3 - Detection:")
print(f"  Mean IoU: {np.mean(metrics['det_iou']):.4f} \u00b1 {np.std(metrics['det_iou']):.4f}")
print(f"\nPipeline Throughput:")
print(f"  Total steps per image: {np.mean(metrics['pipeline_time_steps']):.0f}")

In [ ]:
# Ablation Study: What happens when we remove each stage?
print("Ablation Study: Impact of Each Pipeline Stage")
print("="*60)

ablation_results = {
    'full_pipeline': {'seg_iou': [], 'det_iou': []},
    'no_enhancement': {'seg_iou': [], 'det_iou': []},
    'no_segmentation': {'det_iou': []},
}

for scene in test_scenes[:30]:
    # Full pipeline
    results = pipeline.run(scene)
    ablation_results['full_pipeline']['seg_iou'].append(results['seg_iou'])
    ablation_results['full_pipeline']['det_iou'].append(results['det_iou'])
    
    # Without enhancement (feed degraded directly to segmentation)
    seg_state = seg_env.reset(scene['degraded'], scene['mask'])
    for _ in range(8):
        action, _, _ = seg_policy.get_action(seg_state, deterministic=True)
        seg_state, _, done = seg_env.step(action)
        if done:
            break
    seg_no_enh = seg_env.get_segmentation()
    ablation_results['no_enhancement']['seg_iou'].append(seg_env._compute_iou())
    
    det_state = det_env.reset(seg_no_enh, scene['boxes'])
    for _ in range(12):
        action, _, _ = det_policy.get_action(det_state, deterministic=True)
        det_state, _, done = det_env.step(action)
        if done:
            break
    ablation_results['no_enhancement']['det_iou'].append(det_env._compute_best_iou())
    
    # Without segmentation (use raw threshold for detection)
    state = enhance_env.reset(scene)
    for _ in range(10):
        action, _, _ = enhance_policy.get_action(state, deterministic=True)
        state, _, done = enhance_env.step(action)
        if done:
            break
    enhanced = enhance_env.get_enhanced_image()
    naive_seg = (np.mean(enhanced, axis=2) > 0.4).astype(np.float32)
    
    det_state = det_env.reset(naive_seg, scene['boxes'])
    for _ in range(12):
        action, _, _ = det_policy.get_action(det_state, deterministic=True)
        det_state, _, done = det_env.step(action)
        if done:
            break
    ablation_results['no_segmentation']['det_iou'].append(det_env._compute_best_iou())

# Visualize ablation results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segmentation IoU comparison
conditions = ['Full Pipeline', 'No Enhancement']
seg_means = [np.mean(ablation_results['full_pipeline']['seg_iou']),
             np.mean(ablation_results['no_enhancement']['seg_iou'])]
seg_stds = [np.std(ablation_results['full_pipeline']['seg_iou']),
            np.std(ablation_results['no_enhancement']['seg_iou'])]

bars = axes[0].bar(conditions, seg_means, yerr=seg_stds, 
                   color=['#4ecdc4', '#ff6b6b'], alpha=0.8, capsize=5, edgecolor='black')
axes[0].set_title('Ablation: Segmentation IoU', fontweight='bold')
axes[0].set_ylabel('IoU')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3, axis='y')
for bar, mean in zip(bars, seg_means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{mean:.3f}', ha='center', fontweight='bold')

# Detection IoU comparison
conditions = ['Full Pipeline', 'No Enhancement', 'No Segmentation']
det_means = [np.mean(ablation_results['full_pipeline']['det_iou']),
             np.mean(ablation_results['no_enhancement']['det_iou']),
             np.mean(ablation_results['no_segmentation']['det_iou'])]
det_stds = [np.std(ablation_results['full_pipeline']['det_iou']),
            np.std(ablation_results['no_enhancement']['det_iou']),
            np.std(ablation_results['no_segmentation']['det_iou'])]

bars = axes[1].bar(conditions, det_means, yerr=det_stds,
                   color=['#4ecdc4', '#ff6b6b', '#feca57'], alpha=0.8, capsize=5, edgecolor='black')
axes[1].set_title('Ablation: Detection IoU', fontweight='bold')
axes[1].set_ylabel('Detection IoU')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, mean in zip(bars, det_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{mean:.3f}', ha='center', fontweight='bold')

plt.suptitle('Ablation Study: Contribution of Each Pipeline Stage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nKey findings:")
print(f"  - Enhancement improves segmentation IoU by {seg_means[0]-seg_means[1]:.3f}")
print(f"  - Enhancement improves detection IoU by {det_means[0]-det_means[1]:.3f}")
print(f"  - RL segmentation improves detection IoU by {det_means[0]-det_means[2]:.3f}")

In [ ]:
# Results Gallery: Grid of pipeline outputs
fig, axes = plt.subplots(4, 5, figsize=(18, 14))

col_titles = ['Degraded Input', 'Enhanced', 'Segmented', 'Detected', 'Ground Truth']
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight='bold')

gallery_indices = [155, 165, 175, 185]

for row, idx in enumerate(gallery_indices):
    scene = scenes[idx]
    results = pipeline.run(scene)
    
    axes[row, 0].imshow(results['input'])
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(results['enhanced'])
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(results['segmentation'], cmap='gray')
    axes[row, 2].axis('off')
    
    axes[row, 3].imshow(results['enhanced'])
    box = results['detection']
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                         linewidth=2, edgecolor='red', facecolor='none')
    axes[row, 3].add_patch(rect)
    axes[row, 3].axis('off')
    
    axes[row, 4].imshow(scene['clean'])
    for gt_box in scene['boxes']:
        rect_gt = plt.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                              linewidth=1.5, edgecolor='lime', facecolor='none')
        axes[row, 4].add_patch(rect_gt)
    axes[row, 4].axis('off')

plt.suptitle('Pipeline Results Gallery', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Error Analysis
print("Error Analysis: Where Does the Pipeline Fail?")
print("="*60)

# Categorize results by quality
good_results = []
bad_results = []

for i, scene in enumerate(test_scenes[:30]):
    results = pipeline.run(scene)
    combined_score = results['seg_iou'] * 0.5 + results['det_iou'] * 0.5
    if combined_score > 0.4:
        good_results.append((i, results, scene))
    else:
        bad_results.append((i, results, scene))

print(f"Good results (score > 0.4): {len(good_results)}/{30} ({100*len(good_results)/30:.1f}%)")
print(f"Poor results (score <= 0.4): {len(bad_results)}/{30} ({100*len(bad_results)/30:.1f}%)")

# Analyze failure modes
if bad_results:
    bad_num_objects = [len(s['boxes']) for _, _, s in bad_results]
    good_num_objects = [len(s['boxes']) for _, _, s in good_results]
    print(f"\nAvg objects in good results: {np.mean(good_num_objects):.1f}")
    print(f"Avg objects in poor results: {np.mean(bad_num_objects):.1f}")
    print("\n\u2192 Pipeline tends to struggle with more complex scenes.")

# Visualize good vs bad cases
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

if good_results:
    _, results, scene = good_results[0]
    axes[0, 0].imshow(results['input'])
    axes[0, 0].set_title('Input', fontsize=9)
    axes[0, 1].imshow(results['enhanced'])
    axes[0, 1].set_title(f'Enhanced\nPSNR: {results["psnr"]:.1f}', fontsize=9)
    axes[0, 2].imshow(results['segmentation'], cmap='gray')
    axes[0, 2].set_title(f'Seg IoU: {results["seg_iou"]:.3f}', fontsize=9)
    axes[0, 3].imshow(results['enhanced'])
    box = results['detection']
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                         linewidth=2, edgecolor='red', facecolor='none')
    axes[0, 3].add_patch(rect)
    axes[0, 3].set_title(f'Det IoU: {results["det_iou"]:.3f}', fontsize=9)
    axes[0, 4].imshow(scene['clean'])
    axes[0, 4].set_title('Ground Truth', fontsize=9)
    axes[0, 0].set_ylabel('GOOD\u2713', fontsize=12, color='green', fontweight='bold')

if bad_results:
    _, results, scene = bad_results[0]
    axes[1, 0].imshow(results['input'])
    axes[1, 0].set_title('Input', fontsize=9)
    axes[1, 1].imshow(results['enhanced'])
    axes[1, 1].set_title(f'Enhanced\nPSNR: {results["psnr"]:.1f}', fontsize=9)
    axes[1, 2].imshow(results['segmentation'], cmap='gray')
    axes[1, 2].set_title(f'Seg IoU: {results["seg_iou"]:.3f}', fontsize=9)
    axes[1, 3].imshow(results['enhanced'])
    box = results['detection']
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                         linewidth=2, edgecolor='red', facecolor='none')
    axes[1, 3].add_patch(rect)
    axes[1, 3].set_title(f'Det IoU: {results["det_iou"]:.3f}', fontsize=9)
    axes[1, 4].imshow(scene['clean'])
    axes[1, 4].set_title('Ground Truth', fontsize=9)
    axes[1, 0].set_ylabel('POOR\u2717', fontsize=12, color='red', fontweight='bold')
else:
    axes[1, 0].set_ylabel('(No failures)', fontsize=10)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Error Analysis: Good vs Poor Pipeline Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final comprehensive metrics visualization
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1. PSNR distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(metrics['psnr_before'], bins=15, alpha=0.6, color='#ff6b6b', label='Before', edgecolor='black')
ax1.hist(metrics['psnr_after'], bins=15, alpha=0.6, color='#4ecdc4', label='After', edgecolor='black')
ax1.axvline(np.mean(metrics['psnr_before']), color='#ff6b6b', linestyle='--', linewidth=2)
ax1.axvline(np.mean(metrics['psnr_after']), color='#4ecdc4', linestyle='--', linewidth=2)
ax1.set_title('PSNR Distribution', fontweight='bold')
ax1.set_xlabel('PSNR (dB)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. SSIM distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(metrics['ssim_before'], bins=15, alpha=0.6, color='#ff6b6b', label='Before', edgecolor='black')
ax2.hist(metrics['ssim_after'], bins=15, alpha=0.6, color='#4ecdc4', label='After', edgecolor='black')
ax2.set_title('SSIM Distribution', fontweight='bold')
ax2.set_xlabel('SSIM')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Segmentation IoU distribution
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(metrics['seg_iou'], bins=15, alpha=0.7, color='#45b7d1', edgecolor='black')
ax3.axvline(np.mean(metrics['seg_iou']), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(metrics["seg_iou"]):.3f}')
ax3.set_title('Segmentation IoU Distribution', fontweight='bold')
ax3.set_xlabel('IoU')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Detection IoU distribution
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(metrics['det_iou'], bins=15, alpha=0.7, color='#96ceb4', edgecolor='black')
ax4.axvline(np.mean(metrics['det_iou']), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(metrics["det_iou"]):.3f}')
ax4.set_title('Detection IoU Distribution', fontweight='bold')
ax4.set_xlabel('IoU')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Stage-wise performance radar-style bar plot
ax5 = fig.add_subplot(gs[1, 1])
stage_labels = ['PSNR\nGain (norm)', 'SSIM\nGain (norm)', 'Seg\nIoU', 'Det\nIoU']
psnr_gain_norm = min(1.0, (np.mean(metrics['psnr_after']) - np.mean(metrics['psnr_before'])) / 5.0)
ssim_gain_norm = min(1.0, (np.mean(metrics['ssim_after']) - np.mean(metrics['ssim_before'])) / 0.2)
stage_values = [max(0, psnr_gain_norm), max(0, ssim_gain_norm),
                np.mean(metrics['seg_iou']), np.mean(metrics['det_iou'])]
colors = ['#4ecdc4', '#4ecdc4', '#45b7d1', '#96ceb4']
bars = ax5.bar(stage_labels, stage_values, color=colors, alpha=0.8, edgecolor='black')
ax5.set_title('Normalized Stage Performance', fontweight='bold')
ax5.set_ylim(0, 1.1)
ax5.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, stage_values):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

# 6. Pipeline quality score
ax6 = fig.add_subplot(gs[1, 2])
pipeline_scores = [0.3 * (p/30) + 0.35 * s + 0.35 * d 
                   for p, s, d in zip(metrics['psnr_after'], metrics['seg_iou'], metrics['det_iou'])]
ax6.hist(pipeline_scores, bins=15, alpha=0.7, color='#feca57', edgecolor='black')
ax6.axvline(np.mean(pipeline_scores), color='red', linestyle='--', linewidth=2,
            label=f'Mean: {np.mean(pipeline_scores):.3f}')
ax6.set_title('Overall Pipeline Quality Score', fontweight='bold')
ax6.set_xlabel('Composite Score')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.suptitle('Comprehensive Pipeline Evaluation Metrics', fontsize=14, fontweight='bold')
plt.show()

---

## 9. 🎓 Course Summary & What's Next

### Recap of All Modules

| Module | Topic | Key Technique |
|--------|-------|---------------|
| 1 | Foundations of RL | MDPs, Bellman equations |
| 2 | Image Processing Basics | Filtering, transforms |
| 3 | RL for Enhancement | Policy gradient for brightness/contrast |
| 4 | Value-Based Methods | DQN for denoising |
| 5 | Policy Gradient Methods | REINFORCE, A2C |
| 6 | RL for Segmentation | Threshold optimization |
| 7 | RL for Detection | Bounding box refinement |
| 8 | Multi-Agent Systems | Collaborative processing |
| 9 | Advanced Architectures | CNNs + RL |
| 10 | Reward Engineering | Custom metrics, shaping |
| 11 | Deployment & Optimization | Efficiency, batching |
| **12** | **Real-World Projects** | **End-to-End Pipeline** |

### Key Takeaways

1. **RL enables adaptive processing**: Unlike fixed algorithms, RL agents adapt their
   strategy to each input image
   
2. **Pipeline composition amplifies quality**: Each stage builds on the previous,
   with upstream improvements cascading downstream
   
3. **Reward design is critical**: The right reward function bridges the gap between
   what we can measure and what we want to achieve
   
4. **Trade-offs exist**: More stages add latency but improve quality;
   finding the right balance is key

### Future Directions

- **Larger models**: CNN-based policies for richer state representations
- **Real datasets**: Transfer from synthetic to real-world images
- **Joint training**: True end-to-end gradient flow through all stages
- **Production deployment**: TorchScript/ONNX export, batched inference
- **Meta-learning**: Agents that adapt to new image domains quickly
- **Hierarchical RL**: High-level planner choosing which stages to activate

### Resources for Further Learning

- Sutton & Barto, *Reinforcement Learning: An Introduction* (2018)
- Szeliski, *Computer Vision: Algorithms and Applications* (2022)
- OpenAI Spinning Up in Deep RL
- Papers: "RL for Image Restoration" (Yu et al., 2018), "Attention-driven RL" (Caicedo & Lazebnik, 2015)

---

## 10. 📝 Summary

### What We Built

In this capstone project, we constructed a **complete end-to-end RL vision pipeline** with:

- **Stage 1 (Enhancement)**: An RL agent that learns to restore degraded images through
  sequential application of image processing operations, optimizing PSNR/SSIM
  
- **Stage 2 (Segmentation)**: An RL agent that produces accurate binary segmentation
  masks by adaptively choosing thresholding and morphological operations
  
- **Stage 3 (Detection)**: An RL agent that refines bounding boxes by iteratively
  adjusting position and size to maximize overlap with objects

### Mathematical Framework

The pipeline operates as cascaded MDPs:

$$\mathcal{M}_{pipeline} = \mathcal{M}_1 \circ \mathcal{M}_2 \circ \mathcal{M}_3$$

with joint objective:

$$J(\theta_1, \theta_2, \theta_3) = \mathbb{E}\left[\sum_{k=1}^{3} \alpha_k R_k(s_k, \pi_{\theta_k}(s_k))\right]$$

### Key Results

- Enhancement provides measurable PSNR/SSIM improvements
- Each stage demonstrably benefits from upstream processing (ablation study)
- The pipeline gracefully handles various scene complexities
- REINFORCE with baseline provides stable training across all stages

---

*Congratulations on completing the Reinforcement Learning for Image Processing course!*

*You now have the tools and knowledge to build intelligent, adaptive vision systems
that learn from experience rather than relying on hand-crafted rules.*

---